In [14]:
from tqdm import tqdm
import numpy as np
import h5py
import re
import xml.dom.minidom as minidom
import ismrmrd
import xml.etree.ElementTree as ET
import os


os.chdir(os.path.expanduser("~"))
os.chdir(os.path.join(os.getcwd(), "../.."))

data_folder = os.path.join(os.getcwd(), "data/datasets/msk_mri_h5/h5")
#data_folder = "/home/paula/msk_mri_dataset/metadata_only/meta"
print(f"Total files in folder before cleaning: {len(os.listdir(data_folder))}")
DELETE = True


Total files in folder before cleaning: 2376


# Deleting Files by Name

Certain files can be safely ignored or deleted based on their names:

- **Localizer scans**: If the file name contains terms like "LOC", "Localizer", etc., it is a quick scan used only for patient positioning and not a real scan.
- **Adj files**: If the file name contains "Adj" (e.g., `AdjQuietCoilSens`), these are usually not saved or are redundant noise measurements.
- **Phantom/Test scans**: If the file name contains "_sn", it is likely a phantom or test scan.

These files are not useful for further analysis and can be removed to clean the dataset.

In [17]:
files_to_delete = []

# MIN_SIZE_MB = 100
total_files = 0
# for fname in os.listdir(data_folder):
#     fpath = os.path.join(data_folder, fname)
#     if os.path.isfile(fpath):
#         total_files += 1
#         size_mb = os.path.getsize(fpath) / (1024 * 1024)
#         if size_mb < MIN_SIZE_MB:
#             files_to_delete.append((fname, size_mb))

print(f"Total files: {total_files}")
#print(f"Files under {MIN_SIZE_MB}MB: {len(files_to_delete)}")
if DELETE: 
    for fname, _ in files_to_delete:
        fpath = os.path.join(data_folder, fname)
        os.remove(fpath)
        print(f"Deleted: {fname}")
bad_names = ["adj", "_sn", "loc", "in_out_phase", "semac", "dess", "vibe"]
pattern_files = [fname for fname in os.listdir(data_folder)
                 if (any(bad_name in fname.lower() for bad_name in bad_names) and os.path.isfile(os.path.join(data_folder, fname)))]
print(f"Files containing bad names: {len(pattern_files)}")
if DELETE:
    for fname in pattern_files:
        fpath = os.path.join(data_folder, fname)
        os.remove(fpath)
        print(f"Deleted (bad name): {fname}")


Total files: 0
Files containing bad names: 281
Deleted (bad name): meas_MID00146_FID118947_SAG_T1_IN_OUT_PHASE.h5
Deleted (bad name): meas_MID00289_FID124225_SAG_T1_IN_OUT_PHASE_DIXON_WEAK.h5
Deleted (bad name): meas_MID00310_FID114848_SAG_T1_IN_OUT_PHASE_DIXON_WEAK.h5
Deleted (bad name): meas_MID00380_FID134496_AX_T1_IN_OUT_PHASE_RT_HIP.h5
Deleted (bad name): meas_MID00373_FID117873_SAG_T1_IN_OUT_PHASE.h5
Deleted (bad name): meas_MID00147_FID121429_SAG_T1_IN_OUT_PHASE_DIXON_WEAK.h5
Deleted (bad name): meas_MID00392_FID114436_SAG_T1_IN_OUT_PHASE.h5
Deleted (bad name): meas_MID00167_FID126724_SAG_T1_IN_OUT_PHASE.h5
Deleted (bad name): meas_MID00173_FID125694_SAG_T1_IN_OUT_PHASE.h5
Deleted (bad name): meas_MID00149_FID111117_SAG_T1_IN_OUT_PHASE.h5
Deleted (bad name): meas_MID00387_FID110433_T1_PG_FS_COR_VIBE_3D.h5
Deleted (bad name): meas_MID00316_FID134970_SAG_T1_IN_OUT_PHASE_DIXON_WEAK.h5
Deleted (bad name): meas_MID00423_FID135077_SAG_T1_IN_OUT_PHASE_DIXON_WEAK.h5
Deleted (bad name): 

# Deleting Files by Study Description (`tStudyDescription`)

Some scans are not useful for analysis because they are test, phantom, or quality control scans. These can be identified by specific values in the `tStudyDescription` field of the metadata. If a file's study description matches any of the excluded terms, it should be removed from the dataset.

In [18]:
EXCLUDE_SEQUENCES = [
    # Phantom / QC
    "acr phantom",
    "weekly acr phantom",
    "weekly qc",
    "springbok",

    # Locator / Scout
    "three plane scout",
    "loc",
    "loc uof",
    "cor+-sag loc uof",
    "lt ax loc",
    "rt 3-plane loc uof",
    "l-spine loc uof",
    "haste t-spine loc",

    # Ambiguous / site-specific / test
    "site t2",
    "uof",
    "wp",
    "p2",
    "angle disc",
    "oblique",

    # Generic patterns to catch variations (optional)
    "scout",
    "survey",
    "phantom",
    "qc",
    "localizer",
    "test"
]

all_files = [f for f in os.listdir(data_folder) if os.path.isfile(os.path.join(data_folder, f))]
files_to_delete = []
# ------------------- HELPERS -------------------
def _first_text_from_tag(parent_node, tag_name):
    elems = parent_node.getElementsByTagName(tag_name)
    if elems:
        for node in elems[0].childNodes:
            if node.nodeType == node.TEXT_NODE:
                txt = node.data.strip()
                if txt:
                    return txt
    return None

def read_xml_from_h5(h5_path):
    try:
        with h5py.File(h5_path, "r") as f:
            if "dataset" in f and "xml" in f["dataset"]:
                raw = f["dataset"]["xml"][0]
                if isinstance(raw, (bytes, bytearray)):
                    return raw.decode("utf-8", errors="ignore")
                return str(raw)
    except Exception:
        return None
    return None

def parse_xml_fields(xml_str):
    out = {
        "patientID": None,
        "tStudyDescription": None,
        "patientPosition": None,
        "receiverChannels": None,
        "matrixX": None,
        "matrixY": None,
        "slices": None
    }
    try:
        doc = minidom.parseString(xml_str)
    except Exception:
        return out

    # tStudyDescription
    ups = doc.getElementsByTagName("userParameterString")
    for up in ups:
        name = _first_text_from_tag(up, "name")
        if name == "tStudyDescription":
            out["tStudyDescription"] = _first_text_from_tag(up, "value")
            if out["tStudyDescription"].lower() in EXCLUDE_SEQUENCES:
                files_to_delete.append(fname)
            break

    return out

for fname in all_files:
    xml_str = read_xml_from_h5(os.path.join(data_folder, fname))

print(f"Files to delete by tStudyDescription: {len(files_to_delete)}")

for fname in files_to_delete:
    if DELETE:
        fpath = os.path.join(data_folder, fname)
        os.remove(fpath)
        print(f"Deleted by tStudyDescription: {fname}")

Files to delete by tStudyDescription: 0


# Delete files by protocol name

In [19]:
EXCLUDE_PROTOCOLS = [
    # Locators / scouts / UOF
    "3-plane uof",
    "uof",
    "scout",
    "survey",
    "localizer",
    "loc",

    # Radial / space / exotic (example – adjust as you like)
    "radial",

    # Phantom / QC
    "phantom",
    "qc",

    # Misc test-ish
    "test"
]

# =================== HELPERS ===================

def _first_text_from_tag(parent_node, tag_name):
    elems = parent_node.getElementsByTagName(tag_name)
    if elems:
        for node in elems[0].childNodes:
            if node.nodeType == node.TEXT_NODE:
                txt = node.data.strip()
                if txt:
                    return txt
    return None

def read_xml_from_h5(h5_path):
    try:
        with h5py.File(h5_path, "r") as f:
            if "dataset" in f and "xml" in f["dataset"]:
                raw = f["dataset"]["xml"][0]
                if isinstance(raw, (bytes, bytearray)):
                    return raw.decode("utf-8", errors="ignore")
                return str(raw)
    except Exception:
        return None
    return None

def parse_protocol_name(xml_str):
    """Extract protocolName from measurementInformation, or None."""
    try:
        doc = minidom.parseString(xml_str)
    except Exception:
        return None

    meas_nodes = doc.getElementsByTagName("measurementInformation")
    if not meas_nodes:
        return None

    pname = _first_text_from_tag(meas_nodes[0], "protocolName")
    return pname

def should_exclude_protocol(pname: str) -> bool:
    """Return True if protocolName matches any exclude pattern."""
    if not pname:
        return False
    p_low = pname.lower()
    for pat in EXCLUDE_PROTOCOLS:
        if pat in p_low:
            return True
    return False

# =================== MAIN LOOP ===================

all_files = [
    f for f in os.listdir(data_folder)
    if os.path.isfile(os.path.join(data_folder, f))
]

files_to_delete = []
protocol_hits = []  # (fname, protocolName) for logging/debug

for fname in all_files:
    fpath = os.path.join(data_folder, fname)
    xml_str = read_xml_from_h5(fpath)
    if not xml_str:
        continue

    pname = parse_protocol_name(xml_str)
    if pname and should_exclude_protocol(pname):
        files_to_delete.append(fname)
        protocol_hits.append((fname, pname))

print(f"Total files scanned: {len(all_files)}")
print(f"Files to delete by protocolName: {len(files_to_delete)}\n")

# Optional: print a small log of what matched
for fname, pname in protocol_hits:
    print(f"[MATCH] {fname}  -->  protocolName: {pname}")

if DELETE:
    print("\nDeleting files...")
    for fname in files_to_delete:
        fpath = os.path.join(data_folder, fname)
        try:
            os.remove(fpath)
            print(f"Deleted by protocolName: {fname}")
        except Exception as e:
            print(f"Failed to delete {fname}: {e}")
else:
    print("\nDELETE = False (no files actually removed).")

Total files scanned: 3664
Files to delete by protocolName: 109

[MATCH] meas_MID00179_FID125305_Three_Plane_Scout.h5  -->  protocolName: Three Plane Scout
[MATCH] meas_MID00225_FID118123_RT_3_PLANE_UOF.h5  -->  protocolName: RT 3-PLANE UOF
[MATCH] meas_MID00268_FID128776_Three_Plane_Scout.h5  -->  protocolName: Three Plane Scout
[MATCH] meas_MID00430_FID132058_LT_RADIAL_PD_FS.h5  -->  protocolName: LT RADIAL PD FS
[MATCH] meas_MID00443_FID113914_BIL_HIP_sag+cor__UOF.h5  -->  protocolName: BIL HIP_sag+cor  UOF
[MATCH] meas_MID00165_FID118966_RT_3_PLANE_UOF.h5  -->  protocolName: RT 3-PLANE UOF
[MATCH] meas_MID00112_FID116583_BIL_HIP_sag+cor__UOF.h5  -->  protocolName: BIL HIP_sag+cor  UOF
[MATCH] meas_MID00302_FID126859_BIL_HIP_sag+cor__UOF.h5  -->  protocolName: BIL HIP_sag+cor  UOF
[MATCH] meas_MID00439_FID118770_LT_3_PLANE_UOF.h5  -->  protocolName: LT 3-PLANE UOF
[MATCH] meas_MID00260_FID117760_Three_Plane_Scout.h5  -->  protocolName: Three Plane Scout
[MATCH] meas_MID00092_FID11799

# Add Metadata Fields and Delete by Aspect Ratio

To further clean the dataset, we add new fields to the metadata to record the true shape of the k-space. If the aspect ratio of the scan is less than 1/7, it is likely a line scan and can be deleted. Since the original metadata may not contain the k-space shape, we extract and add this information manually.

In [47]:
# ==== Config ====
from pathlib import Path
import os
import re
import warnings
import numpy as np
from tqdm import tqdm  # <-- keep

import mrpro
from mrpro.data import KData
from mrpro.data.traj_calculators import KTrajectoryCartesian
from mrpro.operators import FastFourierOp

home_folder = os.path.expanduser("~")
parent_folder = os.path.abspath(os.path.join(home_folder, "..", ".."))
ROOT_DIR = Path(parent_folder) / "data/datasets/msk_mri_h5/val_new_metadata"

MIN_SLICES   = 8
MIN_KSPACE_HW = 200
MIN_RECON_HW  = 200

PRINT_EVERY = 10

# ========= Helpers =========
def infer_kspace_dims(kshape):
    if kshape is None or len(kshape) < 4:
        return None, None, None, None
    S, C, H, W = kshape[:4]
    return int(H), int(W), int(S), int(C)

def compute_rss_from_kdata(kdata):
    fft_op = FastFourierOp(
        dim=(-2, -1),
        recon_matrix=kdata.header.recon_matrix,
        encoding_matrix=kdata.header.encoding_matrix,
    )
    (coil_img,) = fft_op.adjoint(kdata.data)   # (S, C, Hy, Hx), complex
    magnitude_fully_sampled = coil_img.abs().square().sum(dim=-4).sqrt().squeeze()
    return magnitude_fully_sampled, coil_img.shape

def is_weird(kH, kW, kS, rH, rW, rS):
    reasons = []
    if kS is not None and kS < MIN_SLICES:
        reasons.append(f"kspace slices={kS} < {MIN_SLICES}")
    if rS is not None and rS < MIN_SLICES:
        reasons.append(f"recon slices={rS} < {MIN_SLICES}")
    if kH is not None and kW is not None and min(kH, kW) < MIN_KSPACE_HW:
        reasons.append(f"kspace min(H,W)={min(kH,kW)} < {MIN_KSPACE_HW}")
    if rH is not None and rW is not None and min(rH, rW) < MIN_RECON_HW:
        reasons.append(f"recon min(H,W)={min(rH,rW)} < {MIN_RECON_HW}")
    return reasons

# Pattern(s) that indicate the “reshape to (acquisitions, coils, 1, 1, k0)” issue
WARNING_PATTERNS = [
    r"reshaped to\s*\(acquisitions,\s*coils,\s*1,\s*1,\s*k0\)",
    r"Found\s*\[\d+,\s*\d+\]\.",  # e.g., "Found [5, 17]."
]

def _warning_matches(msg: str) -> bool:
    return any(re.search(pat, msg, flags=re.IGNORECASE) for pat in WARNING_PATTERNS)

# ========= Main scan =========
h5_files = sorted(Path(ROOT_DIR).rglob("*.h5"))
print(f"Found {len(h5_files)} files under {ROOT_DIR}\n")

weird = []
errors = []

for i, p in enumerate(tqdm(h5_files, total=len(h5_files), desc="Processing files"), 1):
    try:
        # Convert the specific warning(s) into exceptions
        with warnings.catch_warnings(record=True) as wlist:
            warnings.simplefilter("always")
            # Promote our target warning(s) to errors
            warnings.filterwarnings(
                "error",
                message=r".*reshaped to\s*\(acquisitions,\s*coils,\s*1,\s*1,\s*k0\).*",
            )
            # Load kdata (Cartesian)
            kdata = KData.from_file(str(p), KTrajectoryCartesian())

            # K-space shape and dims
            kshape = tuple(kdata.data.squeeze().shape)
            kH, kW, kS, kC = infer_kspace_dims(kshape)

            # Coil image + RSS
            rss, coil_img_shape = compute_rss_from_kdata(kdata)  # rss: (S, Hy, Hx)
            rshape = tuple(rss.shape)
            rS, rH, rW = int(rshape[0]), int(rshape[1]), int(rshape[2])

            # If a non-promoted warning still slipped through, treat as error
            for w in wlist:
                if _warning_matches(str(w.message)):
                    raise RuntimeError(f"mrpro reshape warning treated as error: {w.message}")

        # Weirdness check
        reasons = is_weird(kH, kW, kS, rH, rW, rS)
        if reasons:
            weird.append({
                "path": str(p),
                "kspace_shape": kshape,
                "coil_img_shape": coil_img_shape,
                "rss_shape": rshape,
                "reasons": reasons,
            })

        if PRINT_EVERY and (i % PRINT_EVERY == 0):
            print(f"[{i}/{len(h5_files)}] {p.name}  kspace={kshape}  rss={rshape}, max={rss.max().item():.3f}")

        del rss, kdata

    except Warning as w:
        # Any promoted Warning is treated as error
        errors.append((str(p), f"Warning->Error: {w}"))
        if PRINT_EVERY:
            print(f"[{i}] {p.name}: WARNING->ERROR -> {w}")

    except Exception as e:
        errors.append((str(p), repr(e)))
        if PRINT_EVERY:
            print(f"[{i}] {p.name}: ERROR -> {e}")

# ========= Summary =========
print("\nWeird files:", len(weird))
for e in weird[:200]:
    print(f"- {Path(e['path']).name}: k={e['kspace_shape']}  rss={e['rss_shape']}  -> {', '.join(e['reasons'])}")

if errors:
    print(f"\nErrors: {len(errors)}")
    for fn, msg in errors[:200]:
        print(f"- {Path(fn).name}: {msg}")


Found 394 files under /data/datasets/msk_mri_h5/val_new_metadata



Processing files:   3%|▎         | 10/394 [01:15<47:58,  7.50s/it] 

[10/394] meas_MID00024_FID125150_SAG_PD.h5  kspace=(40, 15, 259, 768)  rss=(40, 384, 384), max=0.000


Processing files:   5%|▌         | 20/394 [02:06<33:06,  5.31s/it]

[20/394] meas_MID00032_FID135148_AX_T2_ANGLE_DISC.h5  kspace=(48, 12, 323, 640)  rss=(48, 320, 320), max=0.000


...

[190/394] meas_MID00254_FID130324_AX_OBL_T2_FS.h5  kspace=(64, 26, 240, 640)  rss=(64, 320, 320), max=0.001


Processing files:  51%|█████     | 200/394 [20:08<21:20,  6.60s/it]

[200/394] meas_MID00259_FID130329_AX_OBL_PD_FS.h5  kspace=(64, 26, 232, 640)  rss=(64, 320, 320), max=0.001


Processing files:  53%|█████▎    | 209/394 [21:18<23:24,  7.59s/it]

: 

In [45]:
from pathlib import Path

targets = [
  "meas_MID00313_FID126870_LT_SAG_PD_FS"
]
parent_folder = os.path.abspath(os.path.join(home_folder, "..", ".."))
ROOT_DIR = Path(parent_folder) / "data/datasets/msk_mri_h5/test_new_metadata"
# Dry run
for name in targets:
    cands = list(ROOT_DIR.rglob(name)) or list(ROOT_DIR.rglob(name + ".h5"))
    if cands:
        for p in cands:
            print("Would remove:", p)
    else:
        print("Not found:", name, "(or with .h5)")

for name in targets:
   cands = list(ROOT_DIR.rglob(name)) or list(ROOT_DIR.rglob(name + ".h5"))
   for p in cands:
       p.unlink(missing_ok=True)
       print("Removed:", p)


Would remove: /data/datasets/msk_mri_h5/test_new_metadata/meas_MID00313_FID126870_LT_SAG_PD_FS.h5
Removed: /data/datasets/msk_mri_h5/test_new_metadata/meas_MID00313_FID126870_LT_SAG_PD_FS.h5


In [1]:
from fastmri.data import SliceDataset
from fastmri.data.subsample import create_mask_for_mask_type
from fastmri.data import transforms as T
import matplotlib.pyplot as plt
import torch
import numpy as np
import os

data_path = '../../../../../data/datasets/msk_mri_h5/varnet_mrpro'
mask_func = create_mask_for_mask_type("equispaced", [0.08], [4])
data_transform = T.VarNetDataTransform(mask_func=mask_func)
dataset = SliceDataset(root=data_path, transform=data_transform, challenge="multicoil")
dataloader = torch.utils.data.DataLoader(dataset, num_workers=8)
broken_files = set()

dataset[21842]

VarNetSample(masked_kspace=tensor([[[[0., 0.],
          [0., 0.],
          [0., 0.],
          ...,
          [0., 0.],
          [0., 0.],
          [0., 0.]],

         [[0., 0.],
          [0., 0.],
          [0., 0.],
          ...,
          [0., 0.],
          [0., 0.],
          [0., 0.]],

         [[0., 0.],
          [0., 0.],
          [0., 0.],
          ...,
          [0., 0.],
          [0., 0.],
          [0., 0.]],

         ...,

         [[0., 0.],
          [0., 0.],
          [0., 0.],
          ...,
          [0., 0.],
          [0., 0.],
          [0., 0.]],

         [[0., 0.],
          [0., 0.],
          [0., 0.],
          ...,
          [0., 0.],
          [0., 0.],
          [0., 0.]],

         [[0., 0.],
          [0., 0.],
          [0., 0.],
          ...,
          [0., 0.],
          [0., 0.],
          [0., 0.]]],


        [[[0., 0.],
          [0., 0.],
          [0., 0.],
          ...,
          [0., 0.],
          [0., 0.],
          [0., 0.]]

In [ ]:
from fastmri.data import SliceDataset
from fastmri.data.subsample import create_mask_for_mask_type
from fastmri.data import transforms as T
import matplotlib.pyplot as plt
import torch
import numpy as np
import os

data_path = '../../../../../data/datasets/msk_mri_h5/varnet_mrpro'
mask_func = create_mask_for_mask_type("equispaced", [0.08], [4])
data_transform = T.VarNetDataTransform(mask_func=mask_func)
dataset = SliceDataset(root=data_path, transform=data_transform, challenge="multicoil")
dataloader = torch.utils.data.DataLoader(dataset, num_workers=8)
broken_files = set()

from tqdm import tqdm

broken_files = set()

for batch, idx in enumerate(dataloader)
    try:
        sample = dataset[idx]
    except Exception as e:
        fname = getattr(sample, "fname", f"idx_{idx}")
        if fname not in broken_files:
            print(f"Broken file at index {idx}: {fname} ({e})")
            broken_files.add(fname)

if not broken_files:
    print("No broken files found.")
else:
    print(f"Broken files: {broken_files}")

Checking slices:  57%|█████▋    | 41603/72832 [42:43<08:08, 63.98it/s]  

Broken file at index 41574: meas_MID00260_FID112237_COR_STIR.h5 ('max')


Checking slices:  59%|█████▉    | 42834/72832 [43:59<06:17, 79.52it/s]  

Broken file at index 42802: meas_MID00265_FID110698_SAG_PD.h5 ('max')


Checking slices:  59%|█████▉    | 43228/72832 [44:13<04:26, 111.27it/s]

Broken file at index 43181: meas_MID00265_FID135381_AX_T2_P2.h5 ('max')


Checking slices: 100%|██████████| 72832/72832 [1:13:49<00:00, 16.44it/s]

Broken files: {'meas_MID00260_FID112237_COR_STIR.h5', 'meas_MID00265_FID135381_AX_T2_P2.h5', 'meas_MID00265_FID110698_SAG_PD.h5'}


In [2]:
# --- Config ---
ROOT = '../../../../../data/datasets/msk_mri_h5/varnet_mrpro'
PATTERN = "*.h5"           # e.g., "*.h5" or "*.hdf5"
WORKERS = 8                # parallel workers (>=1)

# --- Code ---
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor, as_completed
import h5py, sys, os

def _check_one(path_str: str):
    """Return (path, has_attr, error_str_or_None)."""
    try:
        with h5py.File(path_str, "r") as f:
            has_attr = "max" in f.attrs
        return (path_str, has_attr, None)
    except Exception as e:
        return (path_str, False, f"{type(e).__name__}: {e}")

root = Path(os.path.expanduser(ROOT)).resolve()
files = sorted(root.rglob(PATTERN))

if not files:
    print(f"No files found under {root} matching {PATTERN}.")
else:
    results = []
    with ProcessPoolExecutor(max_workers=WORKERS) as ex:
        futs = [ex.submit(_check_one, str(p)) for p in files]
        for fut in as_completed(futs):
            results.append(fut.result())

    # Aggregate
    missing = [p for (p, ok, err) in results if (err is None and not ok)]
    errors  = [(p, err) for (p, ok, err) in results if err is not None]
    present = sum(1 for (_, ok, err) in results if (err is None and ok))

    # Print reports
    if missing:
        print("\nFiles missing f.attrs['max']:\n" + "-"*32)
        for p in sorted(missing):
            print(p)
    else:
        print("\nAll scanned files have f.attrs['max'].")

    if errors:
        print("\nFiles that could not be read:\n" + "-"*32)
        for p, e in sorted(errors):
            print(f"{p}  |  {e}")

    total = len(results)
    print("\nSummary")
    print("-------")
    print(f"Scanned    : {total}")
    print(f"Have 'max' : {present}")
    print(f"Missing    : {len(missing)}")
    print(f"Errors     : {len(errors)}")

    # Keep handy for downstream cells
    CHECK_RESULTS = results
    MISSING_MAX   = missing
    ERROR_FILES   = errors



All scanned files have f.attrs['max'].

Summary
-------
Scanned    : 2407
Have 'max' : 2407
Missing    : 0
Errors     : 0


In [21]:
from pathlib import Path

# === CONFIG ===
FOLDER_A = Path("../../../../../data/datasets/msk_mri_h5/h5")
FOLDER_B = Path("../../../../../data/datasets/msk_mri_h5/varnet_mrpro")
RECURSIVE = False      # set True to include subfolders
COMPARE_STEMS = False  # set True to ignore extensions (compare 'file' vs 'file.jpg')

def list_names(p: Path, recursive=False, stems=False):
    it = p.rglob("*") if recursive else p.iterdir()
    names = []
    for x in it:
        if x.is_file():
            names.append(x.stem if stems else x.name)
    return set(names)

a_names = list_names(FOLDER_A, RECURSIVE, COMPARE_STEMS)
b_names = list_names(FOLDER_B, RECURSIVE, COMPARE_STEMS)

only_in_a = sorted(a_names - b_names)
only_in_b = sorted(b_names - a_names)
in_both   = sorted(a_names & b_names)

print(f"Folder A: {FOLDER_A}")
print(f"Folder B: {FOLDER_B}\n")

print(f"Only in A ({len(only_in_a)}):")
for n in only_in_a:
    print("  ", n)

print(f"\nOnly in B ({len(only_in_b)}):")
for n in only_in_b:
    print("  ", n)



Folder A: ../../../../../data/datasets/msk_mri_h5/h5
Folder B: ../../../../../data/datasets/msk_mri_h5/varnet_mrpro

Only in A (2413):
   meas_MID00015_FID117913_COR_T2.h5
   meas_MID00016_FID118817_COR_T2.h5
   meas_MID00016_FID120712_COR_T2.h5
   meas_MID00017_FID118818_SAG_T2.h5
   meas_MID00017_FID123953_COR_STIR_WP.h5
   meas_MID00017_FID135133_COR_T2.h5
   meas_MID00018_FID110986_COR_PD_FS.h5
   meas_MID00018_FID111995_COR_PD_FS.h5
   meas_MID00018_FID117916_SAG_T2.h5
   meas_MID00018_FID118819_AXIAL_T2.h5
   meas_MID00018_FID123954_COR_T1_WP.h5
   meas_MID00018_FID125144_COR_PD_FS.h5
   meas_MID00019_FID117917_SAG_T1.h5
   meas_MID00019_FID135135_SAG_T2.h5
   meas_MID00020_FID110453_COR_T2.h5
   meas_MID00020_FID110988_AX_PD_FS.h5
   meas_MID00020_FID111997_AX_PD_FS.h5
   meas_MID00020_FID114558_COR_T2.h5
   meas_MID00020_FID115532_COR_T2.h5
   meas_MID00020_FID116491_SAG_STIR.h5
   meas_MID00021_FID117919_SAG_STIR.h5
   meas_MID00021_FID135137_SAG_T1.h5
   meas_MID00022_FID1145

In [13]:
from pathlib import Path
import shutil
import os

# Folder containing images (names correspond to h5 files)
image_folder = Path("/home/paula/msk_mri_dataset/varnet/low_quality")
os.chdir(os.path.expanduser("~"))
os.chdir(os.path.join(os.getcwd(), "../.."))

h5_folder = Path(os.path.join(os.getcwd(), "data/datasets/msk_mri_h5/h5"))

# Destination folder for moving h5 files
dest_folder = Path(os.path.join(os.getcwd(), "data/datasets/msk_mri_h5/low_quality_h5"))
dest_folder.mkdir(exist_ok=True)

# Get image base names (without extension)
# Get all h5 files
h5_files = list(h5_folder.glob("*.h5"))

# Build a set of possible core names from h5 files (without .h5 extension)
h5_core_names = {f.stem for f in h5_files}

# For each image, try to find a matching h5 file by substring
h5_files_to_move = []
for img in image_folder.iterdir():
    if not img.is_file():
        continue
    # Try to find a core h5 name inside the image name
    for core_name in h5_core_names:
        if core_name in img.stem:
            # Find the corresponding h5 file
            h5_file = h5_folder / (core_name + ".h5")
            if h5_file.exists():
                h5_files_to_move.append(h5_file)
            break

print(f"Found {len(h5_files_to_move)} h5 files to move:")


#Move files
for f in h5_files_to_move:
    dst = dest_folder / f.name
    shutil.move(str(f), str(dst))
    print(f"Moved: {f.name} → {dst}")

Found 41 h5 files to move:
Moved: meas_MID00134_FID130204_SAG_STIR.h5 → /data/datasets/msk_mri_h5/low_quality_h5/meas_MID00134_FID130204_SAG_STIR.h5
Moved: meas_MID00108_FID134762_SAG_DBL_OBL_T2_FS.h5 → /data/datasets/msk_mri_h5/low_quality_h5/meas_MID00108_FID134762_SAG_DBL_OBL_T2_FS.h5
Moved: meas_MID00385_FID111353_AX_T1.h5 → /data/datasets/msk_mri_h5/low_quality_h5/meas_MID00385_FID111353_AX_T1.h5
Moved: meas_MID00077_FID111576_COR_T2_FS_DIXON.h5 → /data/datasets/msk_mri_h5/low_quality_h5/meas_MID00077_FID111576_COR_T2_FS_DIXON.h5
Moved: meas_MID00402_FID123768_SAG_STIR.h5 → /data/datasets/msk_mri_h5/low_quality_h5/meas_MID00402_FID123768_SAG_STIR.h5
Moved: meas_MID00252_FID132445_COR_T2_FS.h5 → /data/datasets/msk_mri_h5/low_quality_h5/meas_MID00252_FID132445_COR_T2_FS.h5
Moved: meas_MID00366_FID127415_SAG_T1_PG_FS.h5 → /data/datasets/msk_mri_h5/low_quality_h5/meas_MID00366_FID127415_SAG_T1_PG_FS.h5
Moved: meas_MID00372_FID113843_SAG_STIR.h5 → /data/datasets/msk_mri_h5/low_quality_

In [10]:
names = h5_folder.glob("*.h5")


In [5]:
image_names

{'.DS_Store',
 'ELBOW_meas_MID00077_FID111576_COR_T2_FS_DIXON_grid_with_varnet',
 'HIP_meas_MID00163_FID114207_RT_SAG_PD_FS_grid_with_varnet',
 'HIP_meas_MID00165_FID111664_RT_SAG_PD_FS_grid_with_varnet',
 'HIP_meas_MID00281_FID118612_RT_SAG_PD_FS_grid_with_varnet',
 'HIP_meas_MID00474_FID113945_AX_T1_grid_with_varnet',
 'KNEE_meas_MID00293_FID127342_AX_PD_FS_grid_with_varnet',
 'PELVIS_PUBALGIA_meas_MID00346_FID123712_LT_SAG_PD_FS_grid_with_varnet',
 'SHOULDER_meas_MID00051_FID130121_SAG_T2_FS_grid_with_varnet',
 'SHOULDER_meas_MID00064_FID117962_SAG_DBL_OBL_T2_FS_grid_with_varnet',
 'SHOULDER_meas_MID00068_FID132261_SAG_T2_FS_grid_with_varnet',
 'SHOULDER_meas_MID00072_FID121840_SAG_DBL_OBL_T2_FS_grid_with_varnet',
 'SHOULDER_meas_MID00102_FID121870_SAG_DBL_OBL_T2_FS_grid_with_varnet',
 'SHOULDER_meas_MID00106_FID115057_SAG_DBL_OBL_T2_FS_grid_with_varnet',
 'SHOULDER_meas_MID00108_FID134762_SAG_DBL_OBL_T2_FS_grid_with_varnet',
 'SHOULDER_meas_MID00122_FID114660_SAG_T2_FS_grid_with_va